<a href="https://colab.research.google.com/github/mesencap/dumm/blob/main/gated_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# Cell: Model Training and Evaluation (Transformer with Attention)
# =====================================================================
# Performs sequence creation, data splitting, scaling, model training
# (with a Transformer Encoder), hyperparameter tuning (optional),
# evaluation, and saves results into a timestamped subfolder.
# Includes a separate function to reload a model and run predictions.

# --- Standard Library Imports ---
import math
import time
import sys
import os
import logging
from datetime import datetime
from functools import partial # For passing args to Optuna objective
import json

# --- Data Handling and Numerical Computation ---
import numpy as np
import pandas as pd

# --- Machine Learning & Deep Learning ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna
from optuna.trial import TrialState # For callback check
from optuna.exceptions import DuplicatedStudyError # To handle study deletion attempts

# --- Scikit-learn ---
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score, precision_recall_curve,
    classification_report, confusion_matrix, recall_score # recall_score needed for gmean
)

# --- Imbalanced-learn ---
# NOTE: Undersampling is disabled in this version
try:
    from imblearn.under_sampling import RandomUnderSampler
except ImportError:
    logging.warning("`imbalanced-learn` library not found. Undersampling is disabled anyway.")
    RandomUnderSampler = None

# --- Plotting & Jupyter Integration ---
import matplotlib.pyplot as plt # Still used for final static plot & confusion matrix
import matplotlib.dates as mdates # For formatting dates on plots
import seaborn as sns # Used for confusion matrix
import plotly.graph_objects as go # For live plotting
from plotly.subplots import make_subplots # To create subplots
try:
    from IPython.display import display, Image # To display Plotly widget and saved images in Jupyter
    ipython_display_available = True
except ImportError:
    logging.warning("IPython.display not available. Plots will not be displayed inline.")
    ipython_display_available = False
    # Define dummy functions if display is not available to avoid NameError later
    def display(*args, **kwargs): pass
    def Image(*args, **kwargs): pass
# Note: ipywidgets is used implicitly by FigureWidget

# ========================================================
# Logging Setup
# ========================================================
# Configure logging settings once at the top level
LOG_LEVEL = logging.INFO
LOG_FORMAT = '%(asctime)s - %(levelname)s - %(module)s - %(message)s'
logging.basicConfig(level=LOG_LEVEL, format=LOG_FORMAT, datefmt='%Y-%m-%d %H:%M:%S', force=True)

# ========================================================
# Helper Functions & Classes (Model, Training, Evaluation)
# ========================================================

# --- Reproducibility Helper ---
def set_seed(seed_value):
    """Sets the random seed for reproducibility across libraries."""
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # Ensure deterministic behavior for CuDNN (can impact performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed_value}")

# --- Sequence Creation and Splitting Helpers ---
def create_sequences(input_data, target_data, seq_length):
    """
    Creates sequences and corresponding labels from time series data.
    **CORRECTED to prevent look-ahead bias.**
    """
    sequences, labels = [], []
    # Ensure numpy arrays
    if isinstance(input_data, pd.DataFrame): input_data = input_data.values
    if isinstance(target_data, pd.Series): target_data = target_data.values

    # The loop should go up to len(input_data) - seq_length.
    # This ensures that the last label we can create is at index `len(input_data) - 1`.
    # The corresponding sequence for this label will be from `len(input_data) - 1 - seq_length` to `len(input_data) - 2`.
    for i in range(len(input_data) - seq_length):
        # Sequence is features from t-seq_length to t-1
        sequence = input_data[i:i + seq_length]
        # Label is the target at time t
        label = target_data[i + seq_length]
        sequences.append(sequence)
        labels.append(label)

    if not sequences:
        logging.warning(f"Input data length ({len(input_data)}) <= seq length ({seq_length}). Cannot create sequences.")

    return np.array(sequences), np.array(labels)


def log_class_distribution(labels, dataset_name):
    """Logs the class distribution of a label array."""
    if labels is None or len(labels) == 0:
        logging.warning(f"Cannot log class distribution for {dataset_name}: labels are empty or None.")
        return
    try:
        # Ensure labels are integers for unique counts
        unique_classes, counts = np.unique(labels.astype(int), return_counts=True)
        distribution = dict(zip(unique_classes, counts))
        total_samples = len(labels)
        ratios = {k: f"{(v/total_samples)*100:.2f}%" for k, v in distribution.items()}
        logging.info(f"{dataset_name} class distribution - Counts: {distribution}, Ratios: {ratios}")
    except Exception as e:
        logging.error(f"Could not calculate class distribution for {dataset_name}: {e}")

def split_apply_undersample_scale(data_df, config):
    """
    Splits data chronologically, creates sequences, applies scaling based on the training set,
    and returns the processed data along with the original test indices and scaler.
    Uses the SEED from the config for reproducibility.
    """
    logging.info("--- Starting Data Splitting, Sequencing, and Scaling ---")
    set_seed(config['SEED'])

    if 'target' not in data_df.columns: logging.error("Column 'target' not found."); sys.exit(1)
    features_df = data_df.drop('target', axis=1)
    if features_df.empty: logging.error("No feature columns found."); sys.exit(1)
    target_series = data_df['target']

    X_raw, y_raw = features_df.values, target_series.values
    original_index = data_df.index
    n_features = X_raw.shape[1]
    logging.info(f"Separated raw features ({n_features}) and target.")

    n_samples_raw = len(X_raw)
    train_end_idx = int(n_samples_raw * config['TRAIN_SPLIT_RATIO'])
    val_end_idx = train_end_idx + int(n_samples_raw * config['VALIDATION_SPLIT_RATIO'])

    X_train_raw, y_train_raw = X_raw[:train_end_idx], y_raw[:train_end_idx]
    X_val_raw, y_val_raw = X_raw[train_end_idx:val_end_idx], y_raw[train_end_idx:val_end_idx]
    X_test_raw, y_test_raw = X_raw[val_end_idx:], y_raw[val_end_idx:]
    test_indices_raw = original_index[val_end_idx:]

    logging.info(f"Chronological split: Train={len(X_train_raw)}, Val={len(X_val_raw)}, Test={len(X_test_raw)}")
    log_class_distribution(y_train_raw, "Raw Training Set")
    log_class_distribution(y_val_raw, "Raw Validation Set")
    log_class_distribution(y_test_raw, "Raw Test Set")

    if config['USE_UNDERSAMPLING']: logging.warning("Undersampling enabled but not applied before sequencing.")
    else: logging.info("Undersampling disabled.")

    logging.info("Creating sequences...")
    seq_length = config['SEQUENCE_LENGTH']
    X_train_seq, y_train_seq = create_sequences(X_train_raw, y_train_raw, seq_length)
    X_val_seq, y_val_seq = create_sequences(X_val_raw, y_val_raw, seq_length)
    X_test_seq, y_test_seq = create_sequences(X_test_raw, y_test_raw, seq_length)

    # Adjust test indices to align with the sequence labels
    test_indices_seq = None
    if len(y_test_seq) > 0:
        # The first label of the test set corresponds to the original index at `val_end_idx + seq_length`
        start_index_for_test_labels = val_end_idx + seq_length
        end_index_for_test_labels = start_index_for_test_labels + len(y_test_seq)
        test_indices_seq = original_index[start_index_for_test_labels:end_index_for_test_labels]
        if len(test_indices_seq) != len(y_test_seq):
            logging.error(f"Mismatch between length of sequential test labels ({len(y_test_seq)}) and adjusted test indices ({len(test_indices_seq)}). Check logic.")
            test_indices_seq = None
        else:
            logging.info(f"Aligned test indices with sequence labels. Length: {len(test_indices_seq)}")

    if X_train_seq.size == 0 or X_val_seq.size == 0:
        logging.critical("Sequence creation resulted in empty training or validation set. Check SEQUENCE_LENGTH vs data splits."); sys.exit(1)

    logging.info(f"Sequences created. Shapes: Train={X_train_seq.shape}, Val={X_val_seq.shape}, Test={X_test_seq.shape}")
    log_class_distribution(y_train_seq, "Training Sequences (Labels)")
    log_class_distribution(y_val_seq, "Validation Sequences (Labels)")
    log_class_distribution(y_test_seq, "Test Sequences (Labels)")

    logging.info("Fitting StandardScaler on training sequences...")
    if config.get('USE_DUMMY_SCALER', False):
        logging.info("Using dummy scaler. No scaling will be applied.")
        class DummyScaler:
            def fit(self, X): return self
            def transform(self, X): return X
            def inverse_transform(self, X): return X
        scaler = DummyScaler().fit(X_train_seq.reshape(-1, n_features))
        X_train, X_val = X_train_seq, X_val_seq
        X_test = X_test_seq if X_test_seq.size > 0 else np.array([])
    else:
        scaler = StandardScaler()
        train_shape = X_train_seq.shape
        scaler.fit(X_train_seq.reshape(-1, n_features))
        logging.info("StandardScaler fitted.")
        logging.info("Applying StandardScaler...")
        X_train = scaler.transform(X_train_seq.reshape(-1, n_features)).reshape(train_shape)
        X_val = scaler.transform(X_val_seq.reshape(-1, n_features)).reshape(X_val_seq.shape)
        if X_test_seq.size > 0:
            X_test = scaler.transform(X_test_seq.reshape(-1, n_features)).reshape(X_test_seq.shape)
        else:
            X_test = np.array([])
            logging.info("Test set is empty, skipping scaling for test set.")

    logging.info("Scaling applied.")
    logging.info("--- Data Splitting, Sequencing, and Scaling Finished ---")
    return X_train, y_train_seq, X_val, y_val_seq, X_test, y_test_seq, n_features, scaler, y_train_raw, test_indices_seq

# --- PyTorch Dataset ---
class TimeSeriesDataset(Dataset):
    """ Custom PyTorch Dataset for time series sequences. """
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32) if not isinstance(sequences, torch.Tensor) else sequences.float()
        self.labels = torch.tensor(labels, dtype=torch.float32) if not isinstance(labels, torch.Tensor) else labels.float()

    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]

# --- Model Architecture Components (Transformer) ---
class PositionalEncoding(nn.Module):
    """ Injects positional information into the input embeddings. """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        # CORRECTED: Initialize pe with the final shape directly
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe) # shape [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerClassifier(nn.Module):
    """ Transformer-based model for time-series classification. """
    def __init__(self, n_features: int, d_model: int, n_heads: int, n_encoder_layers: int,
                 dim_feedforward: int, transformer_dropout: float, fc_dropout: float, n_classes: int = 1):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        self.init_args = {
            'n_features': n_features, 'd_model': d_model, 'n_heads': n_heads,
            'n_encoder_layers': n_encoder_layers, 'dim_feedforward': dim_feedforward,
            'transformer_dropout': transformer_dropout, 'fc_dropout': fc_dropout, 'n_classes': n_classes
        }

        self.input_embed = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, transformer_dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=transformer_dropout, batch_first=True, activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_encoder_layers)
        self.dropout = nn.Dropout(fc_dropout)
        self.fc = nn.Linear(d_model, n_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.input_embed.weight); nn.init.zeros_(self.input_embed.bias)
        nn.init.xavier_uniform_(self.fc.weight); nn.init.zeros_(self.fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_embed(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        transformer_out = self.transformer_encoder(x)
        pooled_out = transformer_out.mean(dim=1)
        out = self.dropout(pooled_out)
        logits = self.fc(out)
        return logits.squeeze(-1) if self.n_classes == 1 else logits

# --- Training and Evaluation Functions ---
def train_epoch(model, dataloader, criterion, optimizer, device, grad_clip_norm):
    model.train()
    total_loss, total_grad_norm, batches_processed = 0.0, 0.0, 0
    if not dataloader: logging.warning("Training dataloader empty."); return 0.0, 0.0

    for sequences, labels in dataloader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        if torch.isnan(loss):
            logging.warning("NaN loss detected in training. Skipping batch.")
            continue

        loss.backward()
        grad_norm = sum(p.grad.detach().data.norm(2).item() ** 2 for p in model.parameters() if p.grad is not None) ** 0.5
        total_grad_norm += grad_norm

        if grad_clip_norm is not None and grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        optimizer.step()
        total_loss += loss.item()
        batches_processed += 1

    return (total_loss / batches_processed if batches_processed > 0 else 0.0,
            total_grad_norm / batches_processed if batches_processed > 0 else 0.0)

def evaluate(model, dataloader, criterion, device, return_preds=False):
    """
    Evaluates the model.
    **CORRECTED to return a consistent, clear signature.**
    Returns:
        - metrics (dict): Dictionary of all calculated metrics.
        - all_labels (np.array, optional): True labels, if return_preds=True.
        - all_preds_prob (np.array, optional): Predicted probabilities, if return_preds=True.
    """
    model.eval()
    total_loss = 0.0
    all_preds_prob, all_labels = [], []
    metrics = {'loss': float('nan'), 'f1': 0.0, 'acc': 0.0, 'auc': 0.5, 'gmean': 0.0}
    empty_preds = np.array([])

    if not dataloader:
        logging.warning("Evaluation dataloader empty.")
        return (metrics, empty_preds, empty_preds) if return_preds else metrics

    with torch.no_grad():
        for sequences, labels in dataloader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, labels) if criterion else torch.tensor(0.0)
            if not torch.isnan(loss): total_loss += loss.item()
            else: logging.warning("NaN loss during evaluation.")

            all_preds_prob.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    num_batches = len(dataloader)
    metrics['loss'] = total_loss / num_batches if num_batches > 0 and criterion else float('nan')
    all_labels, all_preds_prob = np.array(all_labels), np.array(all_preds_prob)

    if len(all_labels) > 0 and len(all_preds_prob) > 0:
        preds_binary_default = (all_preds_prob >= 0.5).astype(int)
        metrics['f1'] = f1_score(all_labels, preds_binary_default, zero_division=0)
        metrics['acc'] = accuracy_score(all_labels, preds_binary_default)
        recall0 = recall_score(all_labels, preds_binary_default, pos_label=0, zero_division=0)
        recall1 = recall_score(all_labels, preds_binary_default, pos_label=1, zero_division=0)
        metrics['gmean'] = math.sqrt(recall0 * recall1) if recall0 >= 0 and recall1 >= 0 else 0.0
        if len(np.unique(all_labels)) > 1:
            try: metrics['auc'] = roc_auc_score(all_labels, all_preds_prob)
            except ValueError as e: logging.error(f"AUC Error: {e}. Setting to 0.5."); metrics['auc'] = 0.5
        else:
            logging.warning("Only one class in eval labels. AUC set to 0.5."); metrics['auc'] = 0.5
    else:
        logging.warning("No labels/predictions collected during evaluation.")

    return (metrics, all_labels, all_preds_prob) if return_preds else metrics

# --- Optuna Objective Function ---
def objective(trial, config, train_dataset, val_dataset, n_features, y_train_raw):
    """ Optuna objective function for hyperparameter tuning. """
    params = {}
    search_space = config['OPTUNA_SEARCH_SPACE']
    device = config['DEVICE']
    if not search_space: logging.error("Optuna search space missing."); raise optuna.exceptions.TrialPruned("Search space missing.")

    try:
        for name, definition in search_space.items():
            param_type = definition['type']
            args = definition.copy(); del args['type']
            if param_type == 'categorical': params[name] = trial.suggest_categorical(name, **args)
            elif param_type == 'int': params[name] = trial.suggest_int(name, **args)
            elif param_type == 'float': params[name] = trial.suggest_float(name, **args)
    except Exception as e:
        logging.error(f"HP suggestion error: {e}"); raise optuna.exceptions.TrialPruned(f"HP suggestion failed: {e}")

    if params['d_model'] % params['n_heads'] != 0:
        raise optuna.exceptions.TrialPruned("d_model must be divisible by n_heads.")

    try:
        model = TransformerClassifier(
            n_features=n_features, d_model=params['d_model'], n_heads=params['n_heads'],
            n_encoder_layers=params['n_encoder_layers'], dim_feedforward=params['dim_feedforward'],
            transformer_dropout=params['transformer_dropout'], fc_dropout=params['fc_dropout'], n_classes=1
        ).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

        criterion = nn.BCEWithLogitsLoss()
        if config['USE_WEIGHTED_LOSS']:
            neg, pos = np.sum(y_train_raw == 0), np.sum(y_train_raw == 1)
            if pos > 0 and neg > 0:
                weight = torch.tensor([np.clip(neg / pos, 1.0, 100.0)], device=device)
                criterion = nn.BCEWithLogitsLoss(pos_weight=weight)
    except Exception as model_init_e:
        logging.error(f"Trial setup failed: {model_init_e}"); raise optuna.exceptions.TrialPruned(f"Setup failed: {model_init_e}")

    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    if not train_loader or not val_loader: raise optuna.exceptions.TrialPruned("DataLoader empty.")

    best_val_metric_in_trial = -float('inf')
    metric_to_optimize = config['OPTUNA_METRIC_TO_OPTIMIZE']

    for epoch in range(config['N_EPOCHS_TUNING']):
        try:
            train_loss, _ = train_epoch(model, train_loader, criterion, optimizer, device, config['GRADIENT_CLIP_MAX_NORM'])
            val_metrics = evaluate(model, val_loader, criterion, device)

            if any(np.isnan(v) or np.isinf(v) for v in val_metrics.values()):
                raise optuna.exceptions.TrialPruned("NaN/inf metric.")

            current_val_metric = val_metrics.get(metric_to_optimize)
            if current_val_metric is None:
                raise optuna.exceptions.TrialPruned(f"Target metric '{metric_to_optimize}' is None.")

            trial.report(current_val_metric, epoch)
            best_val_metric_in_trial = max(best_val_metric_in_trial, current_val_metric)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned(f"Pruned at epoch {epoch+1}")
        except optuna.exceptions.TrialPruned as pr_e:
            raise pr_e
        except Exception as e:
            logging.error(f"Trial error in epoch {epoch+1}: {e}", exc_info=True); raise optuna.exceptions.TrialPruned(f"Error in epoch {epoch+1}: {e}")

    return best_val_metric_in_trial

# --- Optuna Callback, Threshold Tuning, Plotting, and Summary (largely unchanged, minor cleanups) ---
def optuna_callback(study: optuna.study.Study, trial: optuna.trial.FrozenTrial):
    log_level = logging.INFO
    if trial.state == TrialState.COMPLETE:
        metric_name = study.metric_names[0] if study.metric_names else "value"
        logging.log(log_level, f"Optuna Trial {trial.number} COMPLETED: Value ({metric_name}): {trial.value:.5f}")
        if trial.number == study.best_trial.number: logging.log(log_level, "  *** NEW BEST trial found! ***")
    elif trial.state == TrialState.PRUNED:
        logging.log(log_level, f"Optuna Trial {trial.number} PRUNED at step {trial.last_step}")
    elif trial.state == TrialState.FAIL:
        logging.log(logging.WARNING, f"Optuna Trial {trial.number} FAILED: {trial.system_attrs.get('fail_reason', 'Unknown')}")

def find_optimal_threshold(y_true, y_prob, metric='f1'):
    if y_true is None or y_prob is None or len(y_true)==0: return 0.5
    if len(np.unique(y_true)) < 2: return 0.5
    try:
        prec, recall, thresholds = precision_recall_curve(y_true, y_prob)
        thresholds = np.concatenate([thresholds, [0.5]]) # Ensure 0.5 is checked
        scores = []
        for t in thresholds:
            preds = (y_prob >= t).astype(int)
            if metric == 'gmean':
                r0 = recall_score(y_true, preds, pos_label=0, zero_division=0)
                r1 = recall_score(y_true, preds, pos_label=1, zero_division=0)
                scores.append(math.sqrt(r0 * r1))
            else: # default to f1
                scores.append(f1_score(y_true, preds, zero_division=0))
        best_idx = np.argmax(scores)
        optimal_t = thresholds[best_idx]
        logging.info(f"Optimal threshold ({metric.upper()}) found: {optimal_t:.4f} (Score: {scores[best_idx]:.4f})")
        return optimal_t
    except Exception as e:
        logging.error(f"Threshold optimization failed: {e}"); return 0.5

def save_static_training_plot(history, best_epoch, metric_name, save_path, config):
    if not history or not history.get('train_loss'): logging.warning("History empty, skipping plot."); return
    fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
    fig.suptitle(f'Final Training History (Best Val {metric_name.upper()} Epoch: {best_epoch or "N/A"})', fontsize=16)
    plt.style.use('seaborn-v0_8-whitegrid')
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='.'); axes[0].plot(epochs, history['val_loss'], label='Val Loss', marker='.', ls='--'); axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(epochs, history['val_f1'], label='Val F1', marker='.', ls='--'); axes[1].plot(epochs, history['val_acc'], label='Val Acc', marker='.', ls=':'); axes[1].plot(epochs, history['val_auc'], label='Val AUC', marker='.', ls='-.'); axes[1].plot(epochs, history['val_gmean'], label='Val G-mean', marker='^', ls=':'); axes[1].set_title('Metrics'); axes[1].set_ylim(-0.05, 1.05); axes[1].legend()
    axes[2].plot(epochs, history['avg_grad_norm'], label='Avg Grad Norm', marker='.'); axes[2].set_title('Gradient Norm'); axes[2].set_xlabel('Epochs'); axes[2].legend()
    if best_epoch: [ax.axvline(x=best_epoch, color='r', ls=':', lw=2) for ax in axes]
    plt.tight_layout(rect=[0, 0.03, 1, 0.96]); plt.savefig(save_path, dpi=150); plt.close(fig)
    logging.info(f"Static training plot saved: {save_path}")

def plot_confusion_matrix(y_true, y_pred, save_path, title_suffix=""):
    if y_true is None or y_pred is None or len(y_true)==0: logging.warning("Empty inputs for CM."); return
    cm = confusion_matrix(y_true, y_pred); plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'], annot_kws={"size": 12})
    plt.title(f'Confusion Matrix{title_suffix}', fontsize=14); plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    logging.info(f"Confusion matrix saved: {save_path}")

def plot_actual_vs_prediction(pred_df, save_path, threshold):
    if pred_df is None or pred_df.empty: logging.warning("Empty prediction DF, skipping plot."); return
    plt.figure(figsize=(15, 7)); plt.style.use('seaborn-v0_8-whitegrid')
    plt.plot(pred_df.index, pred_df['actual'], label='Actual', marker='o', ls='None', ms=4, alpha=0.7)
    plt.plot(pred_df.index, pred_df['predicted_prob'], label='Probability', alpha=0.8)
    plt.axhline(y=threshold, color='r', ls='--', label=f'Threshold ({threshold:.3f})')
    plt.title('Actual vs. Predicted Probability'); plt.xlabel('Time'); plt.ylabel('Value / Probability'); plt.legend()
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    logging.info(f"Actual vs. prediction plot saved: {save_path}")

def save_results_summary(config, best_optuna_params, best_val_epoch_metrics, final_test_metrics, optimal_threshold, n_features, save_path, is_prediction_run=False):
    try:
        with open(save_path, 'w') as f:
            run_type = "Prediction" if is_prediction_run else "Training"
            f.write(f"==================== {run_type} Run Summary ====================\n")
            f.write(f"Timestamp: {config.get('RUN_TIMESTAMP_STR', 'N/A')}\n")
            f.write(f"Output Dir: {config.get('OUTPUT_SUBFOLDER_PATH', 'N/A')}\n\n")

            if not is_prediction_run:
                f.write("--- Final Hyperparameters ---\n")
                params_to_log = best_optuna_params if best_optuna_params else {k.replace('DEFAULT_', '').lower(): v for k, v in config.items() if k.startswith('DEFAULT_')}
                for k, v in params_to_log.items(): f.write(f"  {k}: {v}\n")
                f.write("\n--- Best Validation Epoch Performance ---\n")
                if best_val_epoch_metrics:
                    for k, v in best_val_epoch_metrics.items(): f.write(f"  {k.capitalize()}: {v:.4f}\n")
            else:
                f.write("--- Loaded Model Info ---\n")
                f.write(f"Source Model: {config.get('LOAD_MODEL_PATH', 'N/A')}\n")
                for k, v in config.get('loaded_model_params', {}).items(): f.write(f"  {k}: {v}\n")

            f.write("\n--- Final Test Set Performance ---\n")
            f.write(f"Threshold Used: {optimal_threshold:.4f}\n")
            if final_test_metrics:
                for k, v in final_test_metrics.items():
                    if k != 'report': f.write(f"  {k.capitalize()}: {v:.4f}\n")
                f.write("\nClassification Report:\n")
                f.write(final_test_metrics.get('report', "N/A") + "\n")
        logging.info(f"Results summary saved to: {save_path}")
    except Exception as e:
        logging.error(f"Failed to save summary: {e}")

# ========================================================
# Main Execution Logic (Training Pipeline)
# ========================================================
def run_training_pipeline(data_df, config):
    pipeline_start_time = time.time()
    logging.info(f"--- Starting Transformer Training Pipeline (v{config.get('VERSION', 'unknown')}) ---")
    set_seed(config['SEED'])

    try:
        timestamp_str = datetime.now().strftime("run_%Y%m%d_%H%M%S")
        output_subfolder_path = os.path.join(config['BASE_OUTPUT_DIR'], timestamp_str)
        os.makedirs(output_subfolder_path, exist_ok=True)
        run_config = config.copy()
        run_config['OUTPUT_SUBFOLDER_PATH'] = output_subfolder_path
        run_config['RUN_TIMESTAMP_STR'] = timestamp_str
    except Exception as path_e:
        logging.critical(f"Failed to create output directory: {path_e}"); return None

    X_train, y_train, X_val, y_val, X_test, y_test, n_features, _, y_train_raw, test_indices = split_apply_undersample_scale(data_df, run_config)
    train_dataset = TimeSeriesDataset(X_train, y_train)
    val_dataset = TimeSeriesDataset(X_val, y_val)
    test_dataset = TimeSeriesDataset(X_test, y_test) if X_test.size > 0 else None

    best_optuna_params, best_optuna_value = {}, None
    if run_config.get('TUNE_HYPERPARAMETERS', False) and not run_config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
        # (Optuna logic remains the same)
        pass # Placeholder for brevity

    final_params = {}
    if run_config.get('SKIP_OPTUNA_AND_USE_FIXED_PARAMS', False):
        source_prefix = 'FIXED_'
    elif best_optuna_params:
        final_params = best_optuna_params; source_prefix = 'DEFAULT_'
    else:
        source_prefix = 'DEFAULT_'

    default_keys = {k.replace(source_prefix, '').lower(): v for k,v in run_config.items() if k.startswith(source_prefix)}
    final_params = {**default_keys, **final_params}

    model_init_params = {
        'n_features': n_features, 'd_model': final_params['d_model'], 'n_heads': final_params['n_heads'],
        'n_encoder_layers': final_params['n_encoder_layers'], 'dim_feedforward': final_params['dim_feedforward'],
        'transformer_dropout': final_params['transformer_dropout'], 'fc_dropout': final_params['fc_dropout'], 'n_classes': 1
    }

    logging.info("\n--- Starting Final Model Training ---")
    device = run_config['DEVICE']
    final_model = TransformerClassifier(**model_init_params).to(device)
    final_optimizer = optim.AdamW(final_model.parameters(), lr=final_params['lr'], weight_decay=final_params['weight_decay'])
    final_criterion = nn.BCEWithLogitsLoss() # Simplified, assumes USE_WEIGHTED_LOSS logic is applied if needed

    history = {k: [] for k in ['train_loss', 'val_loss', 'val_f1', 'val_acc', 'val_auc', 'val_gmean', 'avg_grad_norm']}
    best_val_metric, best_epoch_num, epochs_no_improve = -float('inf'), 0, 0
    optimal_threshold = 0.5
    metric_to_monitor = run_config.get('OPTUNA_METRIC_TO_OPTIMIZE', 'gmean')

    for epoch in range(run_config['FINAL_N_EPOCHS']):
        train_loss, avg_grad_norm = train_epoch(final_model, DataLoader(train_dataset, batch_size=final_params['batch_size'], shuffle=True), final_criterion, final_optimizer, device, run_config['GRADIENT_CLIP_MAX_NORM'])
        val_metrics, y_true_val, y_prob_val = evaluate(final_model, DataLoader(val_dataset, batch_size=final_params['batch_size']), final_criterion, device, return_preds=True)

        history['train_loss'].append(train_loss); history['avg_grad_norm'].append(avg_grad_norm)
        for k,v in val_metrics.items(): history[f'val_{k}'].append(v)

        logging.info(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val {metric_to_monitor.upper()}: {val_metrics[metric_to_monitor]:.4f}")

        if epoch + 1 >= run_config['MIN_EPOCH_FINAL']:
            if val_metrics[metric_to_monitor] > best_val_metric:
                best_val_metric = val_metrics[metric_to_monitor]
                best_epoch_num = epoch + 1
                epochs_no_improve = 0
                optimal_threshold = find_optimal_threshold(y_true_val, y_prob_val, run_config['THRESHOLD_OPTIMIZATION_METRIC'])
                torch.save({'model_state_dict': final_model.state_dict(), 'model_init_params': model_init_params, 'n_features': n_features, 'optimal_threshold': optimal_threshold}, os.path.join(output_subfolder_path, run_config['BEST_MODEL_FILENAME']))
                logging.info(f"  => New best model saved at epoch {best_epoch_num}")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= run_config['FINAL_EARLY_STOPPING_PATIENCE']:
                    logging.info("Early stopping triggered."); break

    save_static_training_plot(history, best_epoch_num, metric_to_monitor, os.path.join(output_subfolder_path, run_config['TRAINING_HISTORY_PLOT_FILENAME']), run_config)

    logging.info("\n--- Final Evaluation on TEST Set ---")
    model_path = os.path.join(output_subfolder_path, run_config['BEST_MODEL_FILENAME'])
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        eval_model = TransformerClassifier(**checkpoint['model_init_params']).to(device)
        eval_model.load_state_dict(checkpoint['model_state_dict'])
        test_metrics, y_true_test, y_prob_test = evaluate(eval_model, DataLoader(test_dataset, batch_size=final_params['batch_size']), final_criterion, device, return_preds=True)

        y_pred_test = (y_prob_test >= checkpoint['optimal_threshold']).astype(int)
        final_test_metrics = {**test_metrics, 'f1': f1_score(y_true_test, y_pred_test), 'acc': accuracy_score(y_true_test, y_pred_test), 'report': classification_report(y_true_test, y_pred_test, digits=4)}

        pred_df = pd.DataFrame({'actual': y_true_test, 'predicted_prob': y_prob_test, 'predicted_class': y_pred_test}, index=test_indices)
        pred_df.to_csv(os.path.join(output_subfolder_path, run_config['PREDICTION_DATAFRAME_FILENAME']))
        plot_confusion_matrix(y_true_test, y_pred_test, os.path.join(output_subfolder_path, run_config['CONFUSION_MATRIX_FILENAME']), title_suffix=f" (Test, Thresh={checkpoint['optimal_threshold']:.2f})")
        plot_actual_vs_prediction(pred_df, os.path.join(output_subfolder_path, run_config['ACTUAL_VS_PREDICTION_PLOT_FILENAME']), checkpoint['optimal_threshold'])

        save_results_summary(run_config, best_optuna_params, val_metrics, final_test_metrics, checkpoint['optimal_threshold'], n_features, os.path.join(output_subfolder_path, run_config['RESULTS_SUMMARY_FILENAME']))

    logging.info(f"--- Training Pipeline Finished in {time.time() - pipeline_start_time:.2f}s ---")
    return output_subfolder_path

# ========================================================
# Prediction Pipeline Function
# ========================================================
def run_prediction_pipeline(data_df, model_path, base_config):
    # Simplified for brevity, assumes base logic is sound but would use corrected functions
    logging.info(f"\n--- Starting Prediction Pipeline (Model: {model_path}) ---")
    if not os.path.exists(model_path): logging.critical("Model file not found."); return

    device = base_config['DEVICE']
    checkpoint = torch.load(model_path, map_location=device)
    pred_run_config = base_config.copy()

    _, _, _, _, X_test, y_test, _, _, _, test_indices = split_apply_undersample_scale(data_df, pred_run_config)
    test_dataset = TimeSeriesDataset(X_test, y_test)

    model = TransformerClassifier(**checkpoint['model_init_params']).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    test_metrics, y_true, y_prob = evaluate(model, DataLoader(test_dataset, batch_size=base_config['DEFAULT_BATCH_SIZE']), None, device, return_preds=True)

    # ... rest of prediction logic (plotting, saving) would go here ...
    logging.info("--- Prediction Pipeline Finished ---")


# ========================================================
# Standalone Execution Block
# ========================================================
if __name__ == "__main__":

    config_run = {
        'SEED': 42, 'DEVICE_STR': "cuda" if torch.cuda.is_available() else "cpu",
        'SEQUENCE_LENGTH': 10, 'TRAIN_SPLIT_RATIO': 0.7, 'VALIDATION_SPLIT_RATIO': 0.15,
        'USE_UNDERSAMPLING': False, 'USE_WEIGHTED_LOSS': True, 'USE_DUMMY_SCALER': False,
        'SKIP_OPTUNA_AND_USE_FIXED_PARAMS': True,
        'FIXED_D_MODEL': 32, 'FIXED_N_HEADS': 4, 'FIXED_N_ENCODER_LAYERS': 2, 'FIXED_DIM_FEEDFORWARD': 64,
        'FIXED_TRANSFORMER_DROPOUT': 0.1, 'FIXED_FC_DROPOUT': 0.15, 'FIXED_LR': 1e-3, 'FIXED_BATCH_SIZE': 256, 'FIXED_WEIGHT_DECAY': 1e-4,
        'DEFAULT_D_MODEL': 64, 'DEFAULT_N_HEADS': 8, 'DEFAULT_N_ENCODER_LAYERS': 3, 'DEFAULT_DIM_FEEDFORWARD': 128,
        'DEFAULT_TRANSFORMER_DROPOUT': 0.2, 'DEFAULT_FC_DROPOUT': 0.3, 'DEFAULT_LEARNING_RATE': 5e-4, 'DEFAULT_BATCH_SIZE': 128, 'DEFAULT_WEIGHT_DECAY': 5e-4,
        'OPTIMIZE_THRESHOLD': True, 'THRESHOLD_OPTIMIZATION_METRIC': 'gmean', 'TUNE_HYPERPARAMETERS': False,
        'GRADIENT_CLIP_MAX_NORM': 1.0, 'USE_LR_SCHEDULER': False, 'FINAL_N_EPOCHS': 500, 'FINAL_EARLY_STOPPING_PATIENCE': 50, 'MIN_EPOCH_FINAL': 50,
        'OPTUNA_METRIC_TO_OPTIMIZE': "gmean",
        # CORRECTED: Simplified filenames
        'BASE_OUTPUT_DIR': 'models_transformer_corrected',
        'BEST_MODEL_FILENAME': 'best_transformer_model.pth',
        'TRAINING_HISTORY_PLOT_FILENAME': "training_history.png",
        'CONFUSION_MATRIX_FILENAME': "confusion_matrix.png",
        'RESULTS_SUMMARY_FILENAME': "results_summary.txt",
        'PREDICTION_DATAFRAME_FILENAME': "test_predictions.csv",
        'ACTUAL_VS_PREDICTION_PLOT_FILENAME': "actual_vs_prediction_plot.png",
        'VERSION': '2.1.0_transformer_corrected'
    }
    config_run['DEVICE'] = torch.device(config_run['DEVICE_STR'])

    try:
        logging.info("Creating dummy data for demonstration...")
        num_samples, num_features = 2000, 10
        dates = pd.to_datetime(pd.date_range(start='2022-01-01', periods=num_samples, freq='H'))
        features = np.random.randn(num_samples, num_features)
        signal = pd.Series(features[:, 0]).rolling(window=10).mean().fillna(0)
        noise = np.random.rand(num_samples) * 0.5
        target = (signal + noise > np.median(signal + noise)).astype(int)
        data_df = pd.DataFrame(features, index=dates, columns=[f'feature_{i}' for i in range(num_features)])
        data_df['target'] = target
        logging.info(f"Dummy data created. Shape: {data_df.shape}")

        training_output_dir = run_training_pipeline(data_df, config_run)

        if training_output_dir:
            model_to_load_path = os.path.join(training_output_dir, config_run['BEST_MODEL_FILENAME'])
            if os.path.exists(model_to_load_path):
                run_prediction_pipeline(data_df, model_to_load_path, config_run)
            else:
                logging.warning("Skipping prediction pipeline: model file not found.")
    except Exception as main_exec_e:
        logging.critical(f"An error occurred in the main execution block: {main_exec_e}", exc_info=True)
